# JSALT Multimodal Retrieval Tutorial - Lab Template
# CLIP, CLAP, and Audio-to-Image Retrieval

This lab introduces contrastive multimodal models through three connected tasks. Students will first work with CLIP for image-text alignment, then CLAP for audio-text alignment, and finally combine the two models to build simple audio-to-image retrieval systems.

This is a starter template. The shared overview, setup, and section layout are provided. Each assigned group should fill in the cells in its section with runnable Colab code, concise explanations, small-scale evaluation, and discussion questions.


## Objectives

By the end of this lab, students should be able to:

1. Load pretrained CLIP and CLAP models in a Colab runtime.
2. Preprocess images, audio, and text for contrastive embedding models.
3. Use templated text prompts for zero-shot classification.
4. Use cosine similarity for text-to-image, audio-to-text, and audio-to-image retrieval.
5. Compare direct cross-model embedding retrieval with a text-mediated retrieval pipeline.
6. Explain why two embedding spaces trained with different objectives may not be directly comparable.


## Lab Structure and Assignments

| Part | Topic | Assigned authors | Deliverable |
|---|---|---|---|
| 1 | Work with the CLIP model | Cihan and Zihao | Image-text embedding, zero-shot ImageNet classification, and text-to-image retrieval |
| 2 | Work with the CLAP model | Roger and Chin-Jou | Audio-text embedding, zero-shot ESC-50 classification, and LibriSpeech audio-text similarity |
| 3 | Put CLIP and CLAP together | Jessica, Sandra, and Deb | Audio-to-image retrieval baseline, text-mediated retrieval, and comparison |

Each part should be runnable on a small default subset in Colab. Use configuration values below to keep runtime manageable, then let students increase the sample counts if they have more time or a better GPU.


## Expected Student Flow

1. Run setup and confirm GPU availability.
2. Load the relevant pretrained model and a small subset of the dataset.
3. Inspect a few examples before running evaluation.
4. Embed each modality and compute cosine similarities.
5. Report a simple metric, such as accuracy, recall@k, or paired-retrieval accuracy.
6. Answer the interpretation questions at the end of each part.


## 1. Setup

This lab is intended for Google Colab with a GPU runtime. Click **Runtime** -> **Change runtime type**, then select a GPU accelerator before running the notebook.

The package list below is intentionally broad enough for the three parts. Part authors can simplify it once model choices are finalized.


In [ ]:
!pip install -q torch torchvision torchaudio transformers datasets accelerate evaluate
!pip install -q pillow matplotlib numpy pandas scikit-learn tqdm librosa soundfile

# Optional CLAP package, depending on the implementation chosen in Part 2.
# !pip install -q laion-clap


In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm
from IPython.display import Audio, display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


### 1.1 Check the GPU

A GPU is strongly recommended for the model inference sections. The small default subset sizes should still keep the lab manageable on a T4 runtime.


In [ ]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if "failed" in gpu_info.lower():
    print("Not connected to a GPU")
else:
    print(gpu_info)


## 2. Shared Configuration

Use this dictionary for model names, dataset choices, and small default evaluation sizes. Part authors should update these values when final dataset/model choices are settled.


In [ ]:
config = {
    # Models
    "clip_model_id": "openai/clip-vit-base-patch32",
    "clap_model_id": "laion/clap-htsat-unfused",  # replace if using a different CLAP checkpoint

    # Datasets
    "imagenet_dataset_id": "ILSVRC/imagenet-1k",          # may require HF authentication or a prepared subset
    "esc50_dataset_id": "ashraq/esc50",
    "librispeech_dataset_id": "librispeech_asr",

    # Runtime controls
    "imagenet_subset_size": 200,
    "esc50_subset_size": 200,
    "librispeech_subset_size": 100,
    "batch_size": 32,
    "num_workers": 2,
}

print(config)


## 3. Shared Helper Functions

These helpers should be useful across parts. Extend this section only for utilities that are genuinely shared by multiple parts.


In [ ]:
def normalize_embeddings(x, eps=1e-12):
    """L2-normalize a tensor along its final dimension."""
    return x / x.norm(dim=-1, keepdim=True).clamp_min(eps)


def cosine_similarity_matrix(a, b):
    """Return pairwise cosine similarities between two embedding matrices."""
    a = normalize_embeddings(a)
    b = normalize_embeddings(b)
    return a @ b.T


def topk_accuracy(similarities, targets, k=1):
    """Compute top-k accuracy from a similarity matrix and target column indices."""
    topk = similarities.topk(k, dim=1).indices
    targets = torch.as_tensor(targets, device=topk.device).view(-1, 1)
    return (topk == targets).any(dim=1).float().mean().item()


def show_image_grid(images, titles=None, max_images=8, figsize=(14, 4)):
    """Display a small row of PIL images or image arrays."""
    images = list(images)[:max_images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## 4. Dataset and Model Notes

- **ImageNet:** Full ImageNet access can require credentials or accepting dataset terms. Part 1 should support either a Hugging Face ImageNet validation split or a prepared small subset.
- **ESC-50:** Use animal classes when possible so Part 3 can reuse the same labels.
- **LibriSpeech:** Use a small test split subset for the paired waveform/transcription similarity task.
- **Embedding spaces:** CLIP image/text embeddings and CLAP audio/text embeddings are each internally aligned, but CLIP image embeddings and CLAP audio embeddings are not guaranteed to live in the same semantic space.

Part authors should add any dataset-specific authentication or fallback instructions inside their own sections.


# Part 1: Work with the CLIP Model

**Assigned authors:** Cihan and Zihao

In this part, students use CLIP to connect images and text in a shared embedding space. CLIP has an image encoder and a text encoder. After both outputs are projected into the same dimensionality, cosine similarity can be used as a score for image-text matching.

We will use the Hugging Face `transformers` implementation of `openai/clip-vit-base-patch32`. For a reliable Colab default, this exercise uses Imagenette, a 10-class subset derived from ImageNet. If you have access to the full ImageNet validation set on Hugging Face or a local ImageFolder copy, switch the data source in Section 5.

**Important dataset note:** the official ImageNet test labels are not public. For a tutorial that reports accuracy, use ImageNet validation, a labeled local validation subset, or the Imagenette fallback.

**Exercise format:** code cells marked `### TODO` are for students to complete. The following folded `show solution` cell contains one possible implementation.


## 5. Load CLIP and an ImageNet-Style Evaluation Set

This section loads a pretrained CLIP model and a labeled image dataset. The default dataset source is `imagenette` so the notebook runs without ImageNet credentials. To use a full ImageNet validation set, change `PART1_DATA_SOURCE` to `"hf_imagenet"` and set `config["imagenet_dataset_id"]` to the dataset ID you have access to. To use files uploaded to Colab or mounted from Drive, choose `"local_imagefolder"` and point `LOCAL_IMAGENET_DIR` at a folder laid out like `root/validation/class_name/image.jpg`.

We keep the subset small so students can rerun cells quickly while experimenting with prompts. This loading code is provided because the main learning goals start with preprocessing and similarity scoring.


In [ ]:
from io import BytesIO

from datasets import Dataset, load_dataset
from transformers import AutoProcessor, CLIPModel

# Options:
#   "imagenette"        - small 10-class ImageNet-derived subset; runs without credentials
#   "hf_imagenet"       - full/labeled ImageNet validation split from Hugging Face, if you have access
#   "local_imagefolder" - local files in ImageFolder layout, e.g. root/validation/dog/*.jpg
PART1_DATA_SOURCE = "imagenette"
PART1_SPLIT = "validation"
LOCAL_IMAGENET_DIR = None
PART1_MAX_IMAGES = config["imagenet_subset_size"]

IMAGENETTE_WNID_TO_NAME = {
    "n01440764": "tench",
    "n02102040": "English springer",
    "n02979186": "cassette player",
    "n03000684": "chain saw",
    "n03028079": "church",
    "n03394916": "French horn",
    "n03417042": "garbage truck",
    "n03425413": "gas pump",
    "n03445777": "golf ball",
    "n03888257": "parachute",
}


def clean_class_name(name):
    """Turn dataset label names into short text prompts."""
    name = IMAGENETTE_WNID_TO_NAME.get(str(name), str(name))
    name = name.replace("_", " ")
    return name.split(",")[0].strip()


def get_label_names(dataset):
    label_feature = dataset.features.get("label") if hasattr(dataset, "features") else None
    names = getattr(label_feature, "names", None)
    if names is None:
        return None
    return [clean_class_name(name) for name in names]


def materialize_examples(dataset, max_images, seed=SEED):
    """Return a small list of examples from a Dataset or IterableDataset."""
    if isinstance(dataset, Dataset):
        n = min(max_images, len(dataset))
        subset = dataset.shuffle(seed=seed).select(range(n))
        return [subset[i] for i in range(n)]

    # Streaming datasets do not support random access. Use a shuffle buffer, then take a small subset.
    buffer_size = max(1000, max_images * 10)
    return list(dataset.shuffle(seed=seed, buffer_size=buffer_size).take(max_images))


def as_rgb_image(value):
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, dict) and value.get("bytes") is not None:
        return Image.open(BytesIO(value["bytes"])).convert("RGB")
    if isinstance(value, str):
        return Image.open(value).convert("RGB")
    raise TypeError(f"Unsupported image value type: {type(value)}")


def load_part1_image_data():
    if PART1_DATA_SOURCE == "imagenette":
        dataset = load_dataset("frgfm/imagenette", "320px", split="validation")
        source_desc = "frgfm/imagenette validation split"
    elif PART1_DATA_SOURCE == "hf_imagenet":
        dataset_id = config["imagenet_dataset_id"]
        dataset = load_dataset(dataset_id, split=PART1_SPLIT, streaming=True)
        source_desc = f"{dataset_id} {PART1_SPLIT} split"
    elif PART1_DATA_SOURCE == "local_imagefolder":
        if LOCAL_IMAGENET_DIR is None:
            raise ValueError("Set LOCAL_IMAGENET_DIR before using local_imagefolder.")
        dataset = load_dataset("imagefolder", data_dir=LOCAL_IMAGENET_DIR, split=PART1_SPLIT)
        source_desc = f"local ImageFolder at {LOCAL_IMAGENET_DIR}"
    else:
        raise ValueError(f"Unknown PART1_DATA_SOURCE: {PART1_DATA_SOURCE}")

    label_names = get_label_names(dataset)
    examples = materialize_examples(dataset, PART1_MAX_IMAGES)

    raw_labels = [ex["label"] for ex in examples]
    labels_are_ints = all(isinstance(label, (int, np.integer)) for label in raw_labels)

    if label_names is None:
        if labels_are_ints:
            n_classes = max(int(label) for label in raw_labels) + 1
            label_names = [f"class {i}" for i in range(n_classes)]
        else:
            label_names = list(dict.fromkeys(clean_class_name(label) for label in raw_labels))

    images = [as_rgb_image(ex["image"]) for ex in examples]
    label_ids = []
    for raw_label in raw_labels:
        if isinstance(raw_label, (int, np.integer)):
            label_ids.append(int(raw_label))
        else:
            label_name = clean_class_name(raw_label)
            if label_name not in label_names:
                label_names.append(label_name)
            label_ids.append(label_names.index(label_name))

    labels = [label_names[i] for i in label_ids]
    return images, label_ids, labels, label_names, source_desc


clip_model_id = config["clip_model_id"]
clip_processor = AutoProcessor.from_pretrained(clip_model_id)
clip_model = CLIPModel.from_pretrained(clip_model_id).to(device).eval()

part1_images, part1_label_ids, part1_labels, part1_class_names, part1_source_desc = load_part1_image_data()

print("CLIP model:", clip_model_id)
print("Data source:", part1_source_desc)
print("Images loaded:", len(part1_images))
print("Number of classes:", len(part1_class_names))
print("First classes:", part1_class_names[:10])

show_image_grid(part1_images[:8], part1_labels[:8], max_images=8)


## 6. Preprocess Images and Text

CLIP expects images to be resized, center-cropped, converted to tensors, and normalized with the statistics used during CLIP training. The `AutoProcessor` handles those details.

For labels, we convert each class name into short natural-language prompts. Prompt wording matters: CLIP was trained on image-caption pairs, so `"a photo of a {label}"` is usually better than a bare class name. We use several templates and average their normalized embeddings to get one text embedding per class.


In [ ]:
CLIP_PROMPT_TEMPLATES = [
    "a photo of a {}",
    "a close-up photo of a {}",
    "a cropped photo of a {}",
    "a clear photo of a {}",
]


def batched(items, batch_size):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


@torch.inference_mode()
def encode_clip_images(images, batch_size=None):
    ### TODO 1.1: encode a list of PIL images with CLIP.
    # Steps:
    #   1. Use config["batch_size"] when batch_size is None.
    #   2. Loop over image batches using batched(...).
    #   3. Use clip_processor(images=..., return_tensors="pt") and move inputs to device.
    #   4. Run clip_model.get_image_features(...).
    #   5. L2-normalize the features with normalize_embeddings.
    #   6. Move each batch to CPU and concatenate all batches.
    raise NotImplementedError("Implement image embedding with CLIP")


@torch.inference_mode()
def encode_clip_texts(texts, batch_size=64):
    ### TODO 1.2: encode text strings with CLIP.
    # Steps:
    #   1. Loop over text batches.
    #   2. Use clip_processor(text=..., padding=True, truncation=True, return_tensors="pt").
    #   3. Run clip_model.get_text_features(...).
    #   4. Normalize and concatenate features.
    raise NotImplementedError("Implement text embedding with CLIP")


def make_class_prompts(class_names, templates):
    ### TODO 1.3: create one list of prompts per class name.
    # Example: class "dog" -> ["a photo of a dog", "a close-up photo of a dog", ...]
    raise NotImplementedError("Build templated class prompts")


@torch.inference_mode()
def encode_clip_classes(class_names, templates):
    ### TODO 1.4: create one text embedding per class.
    # Steps:
    #   1. Build prompts for each class.
    #   2. Encode all prompts for a class.
    #   3. Average the prompt embeddings for that class.
    #   4. Normalize the averaged embedding.
    #   5. Stack all class embeddings into one matrix.
    raise NotImplementedError("Implement prompt ensembling for class embeddings")


part1_image_features = encode_clip_images(part1_images)
part1_class_text_features = encode_clip_classes(part1_class_names, CLIP_PROMPT_TEMPLATES)

print("Image feature matrix:", tuple(part1_image_features.shape))
print("Class text feature matrix:", tuple(part1_class_text_features.shape))
print("Example prompts for first class:", make_class_prompts(part1_class_names[:1], CLIP_PROMPT_TEMPLATES)[0])


In [ ]:
#@title show solution { display-mode: "form" }
CLIP_PROMPT_TEMPLATES = [
    "a photo of a {}",
    "a close-up photo of a {}",
    "a cropped photo of a {}",
    "a clear photo of a {}",
]


def batched(items, batch_size):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


@torch.inference_mode()
def encode_clip_images(images, batch_size=None):
    batch_size = batch_size or config["batch_size"]
    all_features = []
    for image_batch in tqdm(list(batched(images, batch_size)), desc="Encoding images"):
        inputs = clip_processor(images=image_batch, return_tensors="pt").to(device)
        features = clip_model.get_image_features(**inputs)
        features = normalize_embeddings(features)
        all_features.append(features.cpu())
    return torch.cat(all_features, dim=0)


@torch.inference_mode()
def encode_clip_texts(texts, batch_size=64):
    all_features = []
    for text_batch in batched(texts, batch_size):
        inputs = clip_processor(text=text_batch, padding=True, truncation=True, return_tensors="pt").to(device)
        features = clip_model.get_text_features(**inputs)
        features = normalize_embeddings(features)
        all_features.append(features.cpu())
    return torch.cat(all_features, dim=0)


def make_class_prompts(class_names, templates):
    return [[template.format(name) for template in templates] for name in class_names]


@torch.inference_mode()
def encode_clip_classes(class_names, templates):
    class_features = []
    for prompts in tqdm(make_class_prompts(class_names, templates), desc="Encoding class prompts"):
        prompt_features = encode_clip_texts(prompts)
        class_feature = normalize_embeddings(prompt_features.mean(dim=0, keepdim=True))[0]
        class_features.append(class_feature)
    return torch.stack(class_features, dim=0)


part1_image_features = encode_clip_images(part1_images)
part1_class_text_features = encode_clip_classes(part1_class_names, CLIP_PROMPT_TEMPLATES)

print("Image feature matrix:", tuple(part1_image_features.shape))
print("Class text feature matrix:", tuple(part1_class_text_features.shape))
print("Example prompts for first class:", make_class_prompts(part1_class_names[:1], CLIP_PROMPT_TEMPLATES)[0])


## 7. Zero-Shot Image Classification

For each image, compare its CLIP image embedding against every class text embedding. The predicted class is the text prompt with the highest cosine similarity.

This is called **zero-shot** classification because CLIP was not trained or fine-tuned on this specific classifier head. The class names and prompt templates define the classifier at inference time.


In [ ]:
### TODO 1.5: classify images by image-text similarity.
# Steps:
#   1. Compute part1_similarity with cosine_similarity_matrix(...).
#   2. Get the highest-scoring class index for each image.
#   3. Convert predicted indices into class names.
#   4. Compute top-1 and top-k accuracy with topk_accuracy(...).
#   5. Build a DataFrame with true label, predicted label, correctness, and score.
#   6. Display the first few rows and per-class accuracy.
#   7. Show a few correct and incorrect examples.

raise NotImplementedError("Implement zero-shot classification with CLIP")


In [ ]:
#@title show solution { display-mode: "form" }
part1_similarity = cosine_similarity_matrix(part1_image_features, part1_class_text_features)
part1_pred_ids = part1_similarity.argmax(dim=1).cpu().tolist()
part1_pred_labels = [part1_class_names[i] for i in part1_pred_ids]

k = min(5, len(part1_class_names))
part1_top1 = topk_accuracy(part1_similarity, part1_label_ids, k=1)
part1_topk = topk_accuracy(part1_similarity, part1_label_ids, k=k)

print(f"Top-1 accuracy on {len(part1_images)} images: {part1_top1:.2%}")
print(f"Top-{k} accuracy on {len(part1_images)} images: {part1_topk:.2%}")

part1_results = pd.DataFrame({
    "true_label": part1_labels,
    "pred_label": part1_pred_labels,
    "correct": [t == p for t, p in zip(part1_labels, part1_pred_labels)],
    "pred_score": [float(part1_similarity[i, pred]) for i, pred in enumerate(part1_pred_ids)],
})

display(part1_results.head(12))
display(part1_results.groupby("true_label")["correct"].agg(["count", "mean"]).sort_values("mean"))

correct_indices = [i for i, ok in enumerate(part1_results["correct"]) if ok]
wrong_indices = [i for i, ok in enumerate(part1_results["correct"]) if not ok]

if correct_indices:
    idxs = correct_indices[:5]
    titles = [f"true: {part1_labels[i]}\npred: {part1_pred_labels[i]}" for i in idxs]
    print("Correct examples")
    show_image_grid([part1_images[i] for i in idxs], titles, max_images=len(idxs), figsize=(14, 3))

if wrong_indices:
    idxs = wrong_indices[:5]
    titles = [f"true: {part1_labels[i]}\npred: {part1_pred_labels[i]}" for i in idxs]
    print("Incorrect examples")
    show_image_grid([part1_images[i] for i in idxs], titles, max_images=len(idxs), figsize=(14, 3))


## 8. Text-to-Image Retrieval

Classification asks: given one image, which label is best? Retrieval asks the reverse kind of question: given a text query, which images score highest?

Encode each query with the CLIP text encoder, compare it to all image embeddings from the subset, and display the top retrieved images. Try changing the queries to use attributes, settings, or visual details rather than class labels only.


In [ ]:
### TODO 1.6: retrieve images from text queries.
# Steps:
#   1. Create a small list of retrieval_queries.
#   2. Encode the queries with encode_clip_texts(...).
#   3. Compare query embeddings to part1_image_features.
#   4. For each query, get the top images with scores.topk(...).
#   5. Display the retrieved images with labels and similarity scores.

raise NotImplementedError("Implement CLIP text-to-image retrieval")


In [ ]:
#@title show solution { display-mode: "form" }
unique_labels_in_subset = list(dict.fromkeys(part1_labels))
retrieval_queries = [f"a photo of a {name}" for name in unique_labels_in_subset[:4]]
retrieval_queries += [
    "a photo taken outdoors",
    "a close-up photo of a round object",
]

query_features = encode_clip_texts(retrieval_queries)
query_similarity = cosine_similarity_matrix(query_features, part1_image_features)

num_to_show = min(5, len(part1_images))
for query, scores in zip(retrieval_queries, query_similarity):
    top_scores, top_indices = scores.topk(num_to_show)
    top_indices = top_indices.cpu().tolist()
    top_scores = top_scores.cpu().tolist()
    titles = [f"{part1_labels[i]}\nscore={score:.2f}" for i, score in zip(top_indices, top_scores)]
    print("Query:", query)
    show_image_grid([part1_images[i] for i in top_indices], titles, max_images=num_to_show, figsize=(15, 3))


## Questions 1.1-1.5

1.1 Run zero-shot classification with only the template `a photo of a {label}`. Then run it with all templates. Does prompt ensembling help on this subset?

1.2 Pick three classes with the lowest accuracy. Are the mistakes visually reasonable, semantically reasonable, or clearly wrong?

1.3 Write two retrieval queries that name an object class and two retrieval queries that describe visual attributes without naming the class. Which kind works better?

1.4 Look at at least five incorrect classification examples. Are any of them ambiguous or mislabeled?

1.5 Why can CLIP do classification without training a new linear classifier on this dataset?


# Part 2: Work with the CLAP Model

**Assigned authors:** Roger and Chin-Jou

**Assignment:**

- Download a CLAP model, ESC-50, and LibriSpeech test data from Hugging Face.
- Teach students how to load and preprocess audio and text.
- Perform zero-shot audio classification on ESC-50 using templated captions.
- Compare CLAP similarities between LibriSpeech utterances and their transcriptions. For a given waveform, test whether CLAP assigns the highest similarity to its paired text transcription compared with other utterance transcriptions.

**Author placeholder:** Replace the cells in this part with runnable Colab code, expected output examples, and 2-4 short student questions.


## 9. Load CLAP, ESC-50, and LibriSpeech

TODO for Part 2 authors:

- Choose the CLAP checkpoint/API and document it.
- Load ESC-50 with labels and audio arrays.
- Load a small LibriSpeech test subset with waveform/transcription pairs.
- Play and plot a few audio examples.


In [ ]:
### TODO Part 2.1
# Load CLAP, ESC-50, and LibriSpeech subsets here.
raise NotImplementedError("Part 2 authors: load CLAP and audio datasets")


## 10. Preprocess Audio and Text

TODO for Part 2 authors:

- Resample audio if the model requires a fixed sample rate.
- Build text prompt templates such as `a sound of a {label}`.
- Encode audio batches and text prompts.
- Verify embedding shapes before evaluation.


In [ ]:
### TODO Part 2.2
# Preprocess audio and text prompts for CLAP.
raise NotImplementedError("Part 2 authors: implement CLAP preprocessing and embedding")


## 11. Zero-Shot Audio Classification on ESC-50

TODO for Part 2 authors:

- Use templated captions for ESC-50 classes.
- Compute audio-text cosine similarities.
- Report accuracy on a small subset and optionally by class.
- Include a few correct and incorrect examples with playable audio.


In [ ]:
### TODO Part 2.3
# Run zero-shot ESC-50 classification with CLAP.
raise NotImplementedError("Part 2 authors: implement zero-shot CLAP classification")


## 12. LibriSpeech Audio-Text Similarity

TODO for Part 2 authors:

- Embed a batch of LibriSpeech waveforms with the CLAP audio encoder.
- Embed the paired transcriptions with the CLAP text encoder.
- Compute the audio-to-text similarity matrix.
- Measure how often the paired transcription is ranked first.
- Discuss what this task says about CLAP's audio-text alignment.


In [ ]:
### TODO Part 2.4
# Compare each LibriSpeech waveform to the batch of text transcriptions.
raise NotImplementedError("Part 2 authors: implement LibriSpeech paired retrieval")


## Questions 2.1-2.4

TODO for Part 2 authors: replace these prompts with final questions.

2.1 Which ESC-50 classes are easiest or hardest for CLAP?

2.2 How does prompt wording affect audio classification?

2.3 Does CLAP reliably match LibriSpeech audio to its paired transcript? Why or why not?

2.4 How is audio-text retrieval different from ASR?


# Part 3: Putting CLIP and CLAP Together

**Assigned authors:** Jessica, Sandra, and Deb

**Goal:** Build an audio-to-image retrieval model. Given an input waveform of an animal sound, such as a dog bark or cat meow, the model should return a picture of the animal that is heard.

**Assignment:**

- Using ImageNet and ESC-50, create image embeddings for several animal classes by taking CLIP embeddings of example ImageNet images.
- Use ESC-50 waveforms for those same animals and use CLAP to embed each waveform.
- Test which animal image embedding has the highest cosine similarity with each audio embedding. Report the percentage of times the correct animal picture is matched to the input audio. This is expected to be low.
- Build a second version that uses CLAP audio-text matching to recognize the animal name, then feeds that text animal name into the CLIP text encoder to select the correct animal image embedding. Report the percentage correct. This is expected to be higher.
- Have students explain why the two audio-to-image retrieval methods perform differently.

**Author placeholder:** Replace the cells in this part with runnable Colab code, expected output examples, and 2-4 short student questions.


## 13. Choose Shared Animal Classes

TODO for Part 3 authors:

- Choose animal classes that exist in both ESC-50 and the ImageNet subset.
- Map dataset labels into a shared label set, such as dog, cat, rooster, cow, frog, and sheep.
- Show example images and play example audio for each class.
- Reuse embeddings from Parts 1 and 2 when possible.


In [ ]:
### TODO Part 3.1
# Define shared animal classes and collect matching image/audio examples.
raise NotImplementedError("Part 3 authors: define aligned animal class subset")


## 14. Baseline: Direct Audio-to-Image Retrieval

TODO for Part 3 authors:

- Embed animal images with the CLIP image encoder.
- Embed animal sounds with the CLAP audio encoder.
- Directly compare the two embedding matrices with cosine similarity.
- Report retrieval accuracy and show example successes/failures.
- Emphasize that this direct comparison is not guaranteed to work because the embeddings come from different models.


In [ ]:
### TODO Part 3.2
# Directly compare CLAP audio embeddings to CLIP image embeddings.
raise NotImplementedError("Part 3 authors: implement direct audio-to-image retrieval")


## 15. Text-Mediated Audio-to-Image Retrieval

TODO for Part 3 authors:

- Use CLAP audio-text similarity to identify the most likely animal label from each audio clip.
- Convert the predicted animal label into a CLIP text prompt.
- Use the CLIP text embedding to retrieve the matching animal image.
- Report retrieval accuracy and compare it with the direct baseline.


In [ ]:
### TODO Part 3.3
# Use text as a bridge from CLAP audio recognition to CLIP image retrieval.
raise NotImplementedError("Part 3 authors: implement text-mediated audio-to-image retrieval")


## 16. Compare the Two Retrieval Pipelines

TODO for Part 3 authors:

- Put direct retrieval and text-mediated retrieval results in one table.
- Visualize several examples side by side.
- Ask students to explain why the text-mediated system should perform better.
- Discuss what would be required to train a model with a shared audio-image embedding space.


In [ ]:
### TODO Part 3.4
# Summarize metrics and visualize example retrievals from both pipelines.
raise NotImplementedError("Part 3 authors: compare direct and text-mediated retrieval")


## Questions 3.1-3.4

TODO for Part 3 authors: replace these prompts with final questions.

3.1 Why is direct comparison between CLAP audio embeddings and CLIP image embeddings expected to be weak?

3.2 Why can text act as a useful bridge between CLAP and CLIP?

3.3 What errors in the CLAP audio-label stage propagate into image retrieval?

3.4 What training data would be needed to make direct audio-to-image retrieval work better?


# Final Reflection

After completing all three parts, students should fill in a short comparison table.

| System | Query modality | Candidate modality | Bridge modality | Metric | Result |
|---|---|---|---|---|---|
| CLIP zero-shot classification | image | text labels | none | top-1 accuracy | TODO |
| CLIP text-to-image retrieval | text | image | none | recall@k or qualitative | TODO |
| CLAP zero-shot classification | audio | text labels | none | top-1 accuracy | TODO |
| CLAP LibriSpeech retrieval | audio | transcript | none | paired rank-1 accuracy | TODO |
| Direct audio-to-image retrieval | audio | image | none | animal retrieval accuracy | TODO |
| Text-mediated audio-to-image retrieval | audio | image | text | animal retrieval accuracy | TODO |

Final discussion prompt: Which systems compare embeddings from the same model, and which systems compare embeddings across different models? How does that affect performance?


## Authoring Checklist

Before the final tutorial is released, confirm that:

- The notebook runs top-to-bottom on a fresh Colab GPU runtime.
- Every default dataset subset is small enough for the tutorial time budget.
- Every TODO placeholder has either been implemented or intentionally left as a student exercise.
- Any required authentication steps are documented before the first failing cell.
- All metrics are printed in clear units.
- The final reflection table can be completed from outputs already produced in the notebook.
